## Segmentation Mask Visualization and Contour Export

This notebook demonstrates how to work with segmentation masks generated by **Cellpose** or **StarDist** in the ISS postprocessing pipeline.

The visualization function supports two image modes via the parameter:

```
input_image_type = "stitched" | "retiled"
```

---

### Workflow

1. **Load Segmentation Mask**  
   The sparse `.npz` mask (e.g., `R1_cellpose_stitched_expanded.npz` or `R1_cellpose_retiled_expanded.npz`) is loaded and converted into a dense label image.

2. **Load Corresponding Raw Image**  
   The raw **DAPI/fluorescence channel** from preprocessing is loaded for visualization.

   - **Stitched mode** loads the image from:  
     ```
     /preprocessing/Cycle1/3_stitched/Cycle1_ch4.tif
     ```

   - **Retiled mode** reconstructs a stitched image from tiles in:  
     ```
     /preprocessing/Cycle1/4_retiled/
     ```
     using the coordinate file:  
     ```
     Cycle1_retiled_coords.csv
     ```

3. **Crop Region of Interest**  
   A subregion of the image can be extracted for inspection (e.g., `4000:8000 × 4000:8000`) to focus on a smaller field of view.

4. **Normalize and Brighten Image**  
   The raw intensities are scaled to `[0, 1]` and enhanced for better contrast.

5. **Overlay Segmentation Boundaries**  
   Cell boundaries from the label mask are overlaid on the brightened raw image using `skimage.segmentation.mark_boundaries`.

6. **Generate Contour Mask**  
   - Outer boundaries are extracted from the label mask with `find_boundaries(mode="outer")`.  
   - Boundaries are thickened via **binary dilation** to make them more visible.  
   - The result is saved as an 8-bit TIFF file (white contours on black background).

7. **Save to Segmentation Folder**  
   The contour mask is written to:

   ```
   <input_dir>/<region>/postprocessing/segmentation/<region>_<segmentation_method>_<input_image_type>_contour_mask.tif
   ```

   ensuring consistency with the pipeline’s folder structure.

### Imports

In [ ]:
import ISS_postprocessing.segmentation as SEG


### Input Parameters

`input_dir` (str): Path to the parent directory containing the **preprocessed region folders**  
(e.g., `/R1/`, `/R2/`, …). These region folders are created automatically during preprocessing.

`region` (str): Region identifier to process (e.g., `"R1"`).

`segmentation_method` (str): Which segmentation mask to use: `"cellpose"` or `"stardist"`.  
Must match the mask file previously generated during segmentation.

`input_image_type` ("stitched" | "retiled", default = `"stitched"`):  
Specifies which segmentation mask and raw image layout to use.

- `"stitched"` → uses segmentation generated from the stitched image  
- `"retiled"` → uses segmentation generated from retiled tiles  

This must match the segmentation mode used during the segmentation step.

`DAPI_ch` (int, default = 4): Index of the **DAPI channel** in the raw images.  
Example: `DAPI_ch = 4` selects `Cycle1_ch4.tif` or `Cycle1_s*_ch4.tif`.

`crop_coords` (tuple of 4 ints, optional): Coordinates for cropping the image and mask before plotting.  
Format: `(y_start, y_end, x_start, x_end)` in pixel coordinates.

Example:

```
crop_coords = (4000, 8000, 4000, 8000)
```

This crops a **4000 × 4000 pixel region**.

If `None` (default), the **full image** is used.  
If the crop extends beyond the image bounds, the full image is shown instead.

---

Note: In this notebook, **only one region is processed at a time**.  
The variable `region` must be a **single string** (e.g., `"R1"`), not a list of regions as in earlier pipeline steps.

In [ ]:
input_dir = '/path/to/regions/'
region = 'R1'


In [ ]:
SEG.inspect_and_work_with_segmentation(
    input_dir,
    region,
    segmentation_method='cellpose',
    DAPI_ch=4,
    input_image_type="stitched",
    crop_coords=None
)